### 연습 문제 
- Doc2Vec를 이용하여 감성 분석
- 데이터는 ratings_train.txt 파일을 로드 
    - 특수 문자, 2칸 이상의 공백의 문자를 제거하는 정규화 함수를 이용
    - document 컬럼의 데이터에서 중복 데이터를 제거 
    - 빈 텍스트가 존재한다면 해당 데이터 제거 
    - 상위 5000개 데이터를 학습 데이터로 이용
- 토큰화 함수로는 Komoran을 사용
    - 품사 필터 : NNP, NNG, VV, VA, MAG, XR 만을 사용
    - 불용어 단어  : 하다, 되다, 이다, 것, 수, 거 단어들을 제외
- 독립변수(document), 종속변수(label) 데이터를 나눠주고 train, text로 데이터를 분할(8:2)
- Doc2Vec 객체를 생성하여 벡터화
    - 매개변수 
        - vector_size = 200
        - window = 5
        - min_count = 2
        - dm = 1
        - negative = 5
        - seed = 42
        - epochs = 50
    - 학습 데이터는 train 데이터를 이용
- Doc2Vec 객체에서 train, test 데이터를 infer_vector() 함수를 이용하여 벡터 데이터를 생성 
- ML 분류 모델을 이용하여 임베딩된 데이터를 독립변수로 사용하여 학습 
    - 정확도를 확인 
    - Logistic 
        - max_iter = 2000
        - random_state = 42
    - LinearSVC
        - random_state = 42
- test데이터를 이용하여 2개의 모델 중 정확도 높은 모델을 검색

In [2]:
import pandas as pd 
import re
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from konlpy.tag import Komoran

In [3]:
df = pd.read_csv("../data/ratings_train.txt", sep = '\t')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        150000 non-null  int64
 1   document  149995 non-null  str  
 2   label     150000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 3.4 MB


In [4]:
df.dropna(inplace=True)

In [13]:
# 텍스트 정규화 함수 
def nomalize(text):
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\.]", ' ', str(text))
    text = re.sub(r"\s+", ' ', text).strip()
    return text

In [7]:
df2 = df.copy()

In [8]:
# document 컬럼의 데이터를 정규화 
df['document'] = df['document'].map(nomalize)

In [ ]:
# df2.apply(lambda x : print(x))

In [21]:
# 실수가 많은 부분 -> df2가 DataFrame에서 Series 변경
df2 = df2['document'].map(nomalize)

In [26]:
# 공백 데이터가 존재하는가? -> 정규화를 통해서 "  " -> " " 이 작업 후 strip()를 사용 -> ""
df = df.loc[
    ~(df['document'] == ''), 
]

In [28]:
# 중복 데이터 제거 
df.drop_duplicates('document', inplace=True)

In [29]:
allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'XG']
stop_word = ['하다', '되다', '이다', '것', '수', '거']

komoran = Komoran()

def tokenize(text):
    tokens = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos and word not in stop_word:
            tokens.append(word)
    return tokens

In [30]:
df3 = df.head(5000)

In [35]:
df3.iloc[2]

id                   10265843
document    너무재밓었다그래서보는것을추천한다
label                       0
Name: 2, dtype: object

In [ ]:
tokenize_sentence = [
    tokenize(text) for text in df3['document'].values 
]
tokenize_sentence

In [36]:
X = tokenize_sentence
y = df3['label'].values

In [38]:
# train, test 데이터셋을 분할 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [39]:
df3['label'].value_counts()

label
0    2505
1    2495
Name: count, dtype: int64

In [ ]:
# 문서에 Tag 부착 
# 빈 토큰 리스트를 제외하고 태그를 부착 
def tagged_docs(token_data):
    tagged = []
    for idx, toks in enumerate(token_data):
        # toks의 길이가 0이라면 -> 학습에서 큰 의미가 없음 제외
        if len(toks) == 0:
            continue

        tagged.append(
            TaggedDocument(
                words = toks, tags = [f"DOC_{idx}"]
            )
        )
    return tagged

In [41]:
X_train_tag = tagged_docs(X_train)
print(len(X_train_tag), len(X_train))

3920 4000


In [42]:
# Doc2Vec 객체를 생성 
model = Doc2Vec(
    documents= X_train_tag, 
    vector_size=200, 
    window = 5, 
    min_count = 2, 
    negative = 5, 
    seed = 42, 
    epochs = 50
)

In [43]:
# Doc2Vec 객체를 먼저 생성하고 추후에 학습 
model2 = Doc2Vec(
    vector_size=200, 
    window = 5, 
    min_count = 2, 
    negative = 5, 
    seed = 42, 
    epochs = 50
)
# 단어 사전 생성 
model2.build_vocab(X_train_tag)
# 학습 
model2.train(
    X_train_tag, total_examples=len(X_train_tag), epochs=50
)

In [44]:
# 2개의 모델에서 단어 사전의 개수를 확인 
print(len(model.wv))
print(len(model2.wv))

2750
2750


In [45]:
print(len(model.dv))

3920


In [46]:
def infer_vector(model, norm_tokens, epochs = 50):
    # norm_texts : 텍스트 정규화가 끝나고 토큰화가 완료된 데이터
    # model : 임베딩 모델 

    result = []

    for tokens in norm_tokens:
        # tokens의 길이가 0이라면 0행렬로 되돌려준다. 
        if len(tokens) == 0:
            result.append(
                np.zeros(model.vector_size, dtype = np.float32)
            )
        else:
            vec = model.infer_vector(tokens, epochs = epochs)
            result.append(vec)
    # return np.array(result)
    return np.vstack(result)

In [48]:
X_train_vec = infer_vector(model, X_train)
X_test_vec = infer_vector(model, X_test)

In [50]:
X_train_vec.shape

(4000, 200)

In [51]:
def eval_clf(model, X_train, X_test, y_train, y_test):
    # model : 분류 모델 입력
    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    print(classification_report(pred, y_test))

In [52]:
logi = LogisticRegression(max_iter=2000, random_state=42)
svc = LinearSVC(random_state=42)

In [53]:
eval_clf(logi, X_train_vec, X_test_vec, y_train, y_test)
eval_clf(svc, X_train_vec, X_test_vec, y_train, y_test)

              precision    recall  f1-score   support

           0       0.73      0.72      0.73       511
           1       0.71      0.73      0.72       489

    accuracy                           0.72      1000
   macro avg       0.72      0.72      0.72      1000
weighted avg       0.72      0.72      0.72      1000

              precision    recall  f1-score   support

           0       0.75      0.73      0.74       512
           1       0.73      0.74      0.73       488

    accuracy                           0.74      1000
   macro avg       0.74      0.74      0.74      1000
weighted avg       0.74      0.74      0.74      1000



In [55]:
# DataFrame을 train, test로 분할 
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

In [56]:
train_df['label'].value_counts()

label
0    58271
1    57515
Name: count, dtype: int64

In [57]:
test_df['label'].value_counts()

label
0    14568
1    14379
Name: count, dtype: int64